In [ ]:
import os 
import numpy as np
import pandas as pd
import json
import re
import ast
from typing import List, Tuple, Dict, Any, Set
import load_dotenv
from openai import OpenAI

# 1. Load dataset

In [ ]:
PATH_JSON = "data/raw_data/mbpp.json"

In [ ]:
# Read JSON string 
json_string = ""
with open(PATH_JSON, 'r') as f:
    json_string = f.read()
    

In [ ]:
data_list = []
for line in json_string.strip().split('\n'):
    if line:  # Skip empty lines
        data_list.append(json.loads(line))
        
df = pd.DataFrame(data_list)
print("Dataset shape:", df.shape)
df.sample()

## 1.1. Explore data

In [ ]:
def extract_function_name(code: str) -> str:
    """Extract the function name from the given code string."""
    match = re.search(r'def\s+(\w+)\s*\(', code)
    if match:
        return match.group(1)
    return ""

In [ ]:
def extract_last_function_signature(code: str):
    """
    Parse the code and return the name and args of the LAST function defined.
    Returns:
        (func_name, arg_list, signature_str)
    """
    tree = ast.parse(code)
    func_nodes = []

    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            func_nodes.append(node)

    if not func_nodes:
        return None, None, None

    # Last function in the script
    last_func = func_nodes[-1]

    func_name = last_func.name
    arg_list = [arg.arg for arg in last_func.args.args]
    signature = f"def {func_name}({', '.join(arg_list)}):"

    return func_name, arg_list, signature

In [ ]:
idx = np.random.randint(0, len(df))
text = df.iloc[idx]['text']
setup_code = df.iloc[idx].get('setup_code', '')
code = df.iloc[idx]['code']
func_name, arg_list, func_signature = extract_last_function_signature(code)
test_list = df.iloc[idx]['test_list']
docs = df.iloc[idx]['docs']

print(f"Text:\n{text}\n")
print(f"Setup Code:\n{setup_code}\n")
print(f"Function Signature:\n{func_signature}\n")
print(f"Code:\n{code}\n")
print(f"Test List:\n{test_list}\n")
print(f"Docs:\n{docs}\n")

## 1.2. Run code evaluation

In this section, we will run through `code` and `test_list` to check the code pass test list

In [ ]:
from typing import List, Tuple, Dict, Any


def run_code_with_tests(
    code: str,
    test_list: List[str],
    setup_code: str = ""
) -> Dict[str, Any]:
    """
    Execute `code` and each statement in `test_list` inside a fresh namespace.

    Returns a dictionary with:
      - total_asserts
      - passed_asserts
      - percentage
      - per_test_results (list)
      - all_passed (bool)
    """

    ns: Dict[str, Any] = {}           # namespace for execution
    per_test_results = []

    # Step 1: Execute setup code
    try:
        if setup_code:
            exec(setup_code, ns, ns)

        exec(code, ns, ns)
    except Exception as e:
        # If code fails to execute, all tests automatically fail
        return {
            "total_asserts": len(test_list),
            "passed_asserts": 0,
            "percentage": 0.0,
            "all_passed": False,
            "per_test_results": [{
                "test": "<initialization>",
                "passed": False,
                "error": repr(e),
            }]
        }

    # Step 2: Run each test
    passed = 0
    total = len(test_list)

    for test_stmt in test_list:
        try:
            exec(test_stmt, ns, ns)  # run the assert statement
            passed += 1
            per_test_results.append({
                "test": test_stmt,
                "passed": True,
                "error": None
            })
        except AssertionError as e:
            per_test_results.append({
                "test": test_stmt,
                "passed": False,
                "error": f"AssertionError: {e}"
            })
        except Exception as e:
            per_test_results.append({
                "test": test_stmt,
                "passed": False,
                "error": repr(e)
            })

    # Step 3: Prepare results
    percentage = passed / total if total > 0 else 0.0
    all_passed = passed == total

    return {
        "total_asserts": total,
        "passed_asserts": passed,
        "percentage": percentage,
        "all_passed": all_passed,
        "per_test_results": per_test_results
    }

In [ ]:
result = run_code_with_tests(
    code=code,
    test_list=test_list,
    setup_code=setup_code
)

print(result["total_asserts"])
print(f"Total asserts: {result['total_asserts']}")
print(f"Passed asserts: {result['passed_asserts']}")
print(f"Pass percentage: {result['percentage']*100:.2f}%")
print(f"All tests passed: {result['all_passed']}")

for r in result["per_test_results"]:
    print(r)

## 1.3. Run whole dataset 

To make sure true `code` is correct

In [ ]:
for idx in range(len(df)):
    text = df.iloc[idx]['text']
    setup_code = df.iloc[idx].get('setup_code', '')
    code = df.iloc[idx]['code']
    test_list = df.iloc[idx]['test_list']

    result = run_code_with_tests(
        code=code,
        test_list=test_list,
        setup_code=setup_code
    )
    if result['all_passed']:
        pass
    else:
        print(f"Sample {idx}: Pass percentage: {result['percentage']*100:.2f}%")

# 2. Using LLM to generate code

In [ ]:
load_dotenv.load_dotenv()

OPEN_AI_API = os.getenv("OPEN_AI_API_v2")
if OPEN_AI_API is None:
    raise ValueError("OPEN_AI_API environment variable not set")
else:
    print("API key loaded successfully")

client = OpenAI(api_key=OPEN_AI_API)

In [ ]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

In [ ]:
idx = np.random.randint(0, len(df))
text = df.iloc[idx]['text']
setup_code = df.iloc[idx].get('setup_code', '')
code = df.iloc[idx]['code']
name, args, func_signature = extract_last_function_signature(code)
test_list = df.iloc[idx]['test_list']

print(f"Text:\n{text}\n")
print(f"Setup Code:\n{setup_code}\n")
print(f"Function Signature:\n{func_signature}\n")
print(f"Code:\n{code}\n")
print(f"Test List:\n{test_list}\n")

In [ ]:
constraints = """
Output only a complete and valid Python code for this function. 
Do not add more explanations or surrounding text and Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

if setup_code:
    if setup_code != "":
        text += f". Please notice the setup code:\n{setup_code}\n"
        
if func_signature:
    text += f". The function signature is '{func_signature}'"

input_prompt = f"""write a complete python function
based on the following description:\n{text}.\n
with the following constraints:{constraints}
"""

print("Input prompt to GPT-4:")
print(input_prompt)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-4o" 
    messages=[
        {"role": "system", "content": "You are an expert in Python."},
        {"role": "user", "content": input_prompt},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

In [ ]:
generated_code = extract_function(output)
print(f"The generated code:\n")
print(generated_code)

## 2.1.  Evaluate the generated code

In [ ]:
result = run_code_with_tests(
    code=generated_code,
    test_list=test_list,
    setup_code=setup_code
)

print(result["total_asserts"])
print(f"Total asserts: {result['total_asserts']}")
print(f"Passed asserts: {result['passed_asserts']}")
print(f"Pass percentage: {result['percentage']*100:.2f}%")
print(f"All tests passed: {result['all_passed']}")

for r in result["per_test_results"]:
    print(r)

# 3. Run whole data

In [ ]:
constraints = """
    Output only a complete and valid Python code for this function. 
    Do not add more explanations or surrounding text and Do not change the provided function signature.
    Wrap your output strictly between the markers:
    <code>
    ... your code ...
    </code>
    """

In [ ]:
output_list = []

for idx in range(len(df)):
    try:
        text = df.iloc[idx]['text']
        setup_code = df.iloc[idx].get('setup_code', '')
        code = df.iloc[idx]['code']
        name, args, func_signature = extract_last_function_signature(code)
        test_list = df.iloc[idx]['test_list']

        if setup_code:
            if setup_code != "":
                text += f". Please notice the setup code:\n{setup_code}\n"
                
        if func_signature:
            text += f". The function signature is '{func_signature}'"

        input_prompt = f"""write a complete python function
        based on the following description:\n{text}.\n
        with the following constraints:{constraints}
        """

        response = client.chat.completions.create(
            model="gpt-4o",  # or "gpt-4o" 
            messages=[
                {"role": "system", "content": "You are an expert in Python."},
                {"role": "user", "content": input_prompt},
            ],
        )

        output = response.choices[0].message.content
        generated_code = extract_function(output)

        result = run_code_with_tests(
            code=generated_code,
            test_list=test_list,
            setup_code=setup_code
        )
        total_asserts = result["total_asserts"]
        passed_asserts = result["passed_asserts"]
        percentage = result["percentage"]
        all_passed = result["all_passed"]

        output_list.append({
            "text": text,
            "setup_code": setup_code,
            "func_name": func_name,
            "code": code,
            "test_list": test_list,
            "generated_code": generated_code,
            "total_asserts": total_asserts,
            "passed_asserts": passed_asserts,
            "percentage": percentage,
            "all_passed": all_passed,
        })
    except Exception as e:
        print(f"Error processing sample {idx}: {e}")
        pass

output_df = pd.DataFrame(output_list)
print("Output DataFrame shape:", output_df.shape)

output_df.to_csv("data/generated/mbpp_generated_gpt4o.csv", index=False)
print(f"[DONE] Saved to data/generated/mbpp_generated_gpt4o.csv")

In [ ]:
average_percentage = output_df['percentage'].mean()
print(f"Average pass percentage over evaluated samples: {average_percentage*100:.2f}%")

## 3.1. Test again generated code

In [ ]:
idx = np.random.randint(0, len(output_df))
text = output_df.iloc[idx]['text']
setup_code = output_df.iloc[idx]['setup_code']
generated_code = output_df.iloc[idx]['generated_code']

true_code = df.iloc[idx]['code']

test_list = output_df.iloc[idx]['test_list']

print(f"Text:\n{text}\n")
print(f"Setup Code:\n{setup_code}\n")
print(f"Generated Code:\n{generated_code}\n")
print(f"Test List:\n{test_list}\n")
print('-'*80)
print(f"True Code:\n{true_code}\n")

In [ ]:
result = run_code_with_tests(
    code=generated_code,
    test_list=test_list,
    setup_code=setup_code
)

print(result["total_asserts"])
print(f"Total asserts: {result['total_asserts']}")
print(f"Passed asserts: {result['passed_asserts']}")
print(f"Pass percentage: {result['percentage']*100:.2f}%")
print(f"All tests passed: {result['all_passed']}")

for r in result["per_test_results"]:
    print(r)